# Explainable AI for Solar PV Soiling: Discovering Physical Tipping Points

## Research Objectives
1. **The Cementation Boundary:** Uncover the exact non-linear threshold where ambient moisture (Relative Humidity / near-zero Dew Deficit) interacts with Particulate Matter ($PM_{10}$ / $PM_{2.5}$) to transform loose dust into cemented crust on PV glass.
2. **The Critical Wind Resuspension Threshold ($v_{crit}$):** Identify the critical aerodynamic velocity threshold where wind ceases to act as a dust depositor and begins functioning as a natural self-cleaning agent.
3. **Operational Dispatch System:** Ingest forward weather forecasts to trigger preventive cleaning before cementation and delay robotic cleaning to harness free wind cleaning.

In [ ]:
import sys
from pathlib import Path

# Add repository root
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import PVDataLoader
from src.feature_engineering import FeatureEngineer
from src.train import ModelTrainer
from src.xai_engine import XAIEngine
from src.threshold_miner import ThresholdMiner
from src.operational_engine import OperationalEngine

sns.set_theme(style="whitegrid", font_scale=1.1)
print("Libraries successfully imported.")

## 1. Data Ingestion & Physical Schema Harmonization
Load data from `data/raw/` (or synthesize physics-informed benchmark if empty).

In [ ]:
loader = PVDataLoader(raw_dir="../data/raw", processed_dir="../data/processed")
raw_df = loader.load_data(auto_generate_if_empty=True)
print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head()

## 2. Mechanistic Feature Engineering
- **Magnus-Tetens Dew-Point:** $T_{dew}$ and $Dew\_Deficit = T_{ambient} - T_{dew}$
- **Soiling Ratio & Step-Change Target:** $SR_t = P_{soiled} / P_{clean}^{corr}$, $\Delta SR_t = SR_t - SR_{t-1}$
- **Cumulative Exposures:** $\sum PM_{10} \cdot dt$, hours since rain ($Rain > 1.0\text{ mm}$)
- **Aerodynamic Proxies:** Shear stress proxy ($v^2$) and kinetic flux proxy ($v^3$)

In [ ]:
fe = FeatureEngineer()
featured_df = fe.extract_features(raw_df, compute_targets=True)
feature_cols = fe.get_feature_names()
print(f"Total engineered features: {len(feature_cols)}")
featured_df[["datetime", "soiling_ratio", "delta_soiling_ratio", "temp_dew", "dew_deficit", "nocturnal_condensation_flag"]].head()

In [ ]:
# Plot Soiling Ratio Dynamics and Rain Washing Events
fig, ax1 = plt.subplots(figsize=(14, 5), dpi=150)
ax2 = ax1.twinx()

sample_slice = featured_df.iloc[:24*60]  # First 60 days
ax1.plot(sample_slice["datetime"], sample_slice["soiling_ratio"], color="#1f77b4", lw=1.8, label="Soiling Ratio (SR)")
ax2.bar(sample_slice["datetime"], sample_slice["rainfall"], color="#2ca02c", alpha=0.5, width=0.04, label="Rainfall (mm)")

ax1.set_ylabel("Soiling Ratio (SR)", color="#1f77b4", fontweight="bold")
ax2.set_ylabel("Rainfall (mm)", color="#2ca02c", fontweight="bold")
ax1.set_title("PV Soiling Degradation, Cementation Drops & Rain Cleaning Restoration", fontweight="bold", pad=12)
plt.show()

## 3. Model Training & Purged Walk-Forward Cross-Validation
Train gradient-boosted trees (LightGBM and XGBoost) on target $\Delta SR_t$ with Purged Time-Series CV.

In [ ]:
trainer = ModelTrainer(output_dir="../models")
X, y, clean_df = trainer.prepare_dataset(
    featured_df,
    feature_cols=feature_cols,
    target_col="delta_soiling_ratio",
    filter_daylight_only=True
)

champion_type, champion_model, champion_metrics = trainer.compare_and_select_best(
    X, y, clean_df, target_name="delta_soiling_ratio"
)
print(f"Selected Champion: {champion_type.upper()} | RMSE: {champion_metrics.rmse:.6f} | R2: {champion_metrics.r2:.4f}")

## 4. Explainable AI: SHAP Global Attribution
Inspect the physical hierarchy of drivers influencing soiling step-changes.

In [ ]:
xai = XAIEngine(model=champion_model, feature_names=list(X.columns), output_dir="../figures")
shap_values = xai.compute_shap_values(X, max_samples=400)

import shap
plt.figure(figsize=(10, 7), dpi=150)
shap.summary_plot(shap_values, xai.sample_df, feature_names=list(X.columns), max_display=12, show=True)

## 5. Mining the Wind Resuspension Boundary ($v_{crit}$)
Detect the critical zero-crossing point where wind velocity transitions from dust deposition to aerodynamic lift.

In [ ]:
miner = ThresholdMiner(smoothing_window=15)
wind_vals, wind_shap = xai.get_1d_dependence_curve("wind_speed")
v_crit, zero_cross, knee_pt = miner.mine_wind_resuspension_threshold(wind_vals, wind_shap)

print(f"Discovered Resuspension Threshold: v_crit = {v_crit:.2f} m/s")
print(f"  - Zero-crossing (net positive effect): {zero_cross:.2f} m/s")
print(f"  - Knee inflection: {knee_pt:.2f} m/s")

# Display Wind Dependence with Annotated Regime
fig_path = xai.plot_wind_dependence(v_crit=v_crit)
from IPython.display import Image
Image(filename=str(fig_path))

## 6. Mining the Cementation Cliff: 2D SHAP Interactions
Inspect the non-linear interaction between atmospheric moisture and particulate matter.

In [ ]:
xai.compute_interaction_values(X, max_samples=350)
cementation_path = xai.plot_2d_cementation_cliff(
    feat_x="relative_humidity",
    feat_y="pm10",
    save_name="shap_cementation_cliff_rh_pm10.png"
)
Image(filename=str(cementation_path))

In [ ]:
dew_deficit_path = xai.plot_2d_cementation_cliff(
    feat_x="dew_deficit",
    feat_y="pm2_5",
    save_name="shap_cementation_cliff_dew_deficit_pm25.png"
)
Image(filename=str(dew_deficit_path))

## 7. Operational Cleaning Dispatch Engine
Ingest a 48h weather forecast to test dispatch recommendations:
- `PREVENTIVE_WASH_BEFORE_DEW`: Wash panels before nocturnal condensation bonds PM.
- `DEFER_WASH_FOR_WIND_CLEANING`: Cancel scheduled robotic wash if dry wind $> v_{crit}$ will clean array for free.

In [ ]:
tp = miner.mine_all_thresholds(
    wind_speeds=wind_vals,
    wind_shap=wind_shap,
    dew_deficits=xai.get_1d_dependence_curve("dew_deficit")[0],
    pm10_vals=xai.interaction_sample_df["pm10"].values,
    interaction_dew_pm=xai.extract_2d_interaction("dew_deficit", "pm10")[2],
    rh_vals=xai.get_1d_dependence_curve("relative_humidity")[0],
    rh_shap=xai.get_1d_dependence_curve("relative_humidity")[1]
)

operational_engine = OperationalEngine(tipping_points=tp, fe=fe, output_dir="../reports")
forecast_df = operational_engine.generate_sample_forecast(hours=48)
decisions, summary = operational_engine.evaluate_forecast(forecast_df)

print("=== Operational Decision Summary ===")
for k, v in summary["operational_highlights"].items():
    print(f"{k}: {v}")

# Display preview of dispatch table
dispatch_df = pd.DataFrame([vars(d) for d in decisions])
dispatch_df[["timestamp", "cementation_risk_level", "cementation_risk_score", "free_cleaning_predicted", "recommended_action"]].head(10)